# Tutorial 1: Orbital Fault Injection

**Objective:** Understand how space radiation causes bit-flips in neural network weights and measure the impact on inference accuracy.

## Background

In Low Earth Orbit (LEO), satellites are exposed to galactic cosmic rays and trapped particles in the Van Allen belts. These particles can cause **Single Event Upsets (SEUs)** — bit-flips in memory — that corrupt model weights during inference.

In this tutorial you will:
1. Configure a satellite orbit and radiation environment
2. Inject radiation-induced faults into a neural network
3. Measure accuracy degradation as fault count increases
4. Visualize the per-layer sensitivity heatmap

In [ ]:
import torch
import torch.nn as nn
import copy

from space_ml_sim.core.orbit import OrbitConfig
from space_ml_sim.environment.radiation import RadiationEnvironment
from space_ml_sim.compute.fault_injector import FaultInjector
from space_ml_sim.models.chip_profiles import TRILLIUM_V6E, RAD5500

## Step 1: Define an orbit

We'll use a 550 km sun-synchronous orbit — similar to Starlink satellites.

In [ ]:
orbit = OrbitConfig(
    altitude_km=550,
    inclination_deg=97.6,  # SSO
    raan_deg=0.0,
    true_anomaly_deg=0.0,
)

rad_env = RadiationEnvironment.from_orbit(orbit)
print(f"SEU rate: {rad_env.seu_rate_per_bit_per_day:.2e} bit-flips/bit/day")
print(f"TID rate: {rad_env.tid_rate_rad_per_day:.2f} rad/day")

## Step 2: Create a simple model

We'll use a small CNN for CIFAR-10-scale classification.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 4 * 4, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

model = SimpleCNN().eval()
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## Step 3: Fault sweep

Inject increasing numbers of bit-flips and measure how predictions change.

In [ ]:
injector = FaultInjector(rad_env=rad_env, chip_profile=TRILLIUM_V6E)

# Generate a fixed test batch
torch.manual_seed(42)
test_input = torch.randn(32, 3, 32, 32)

# Baseline predictions
with torch.no_grad():
    baseline_preds = model(test_input).argmax(dim=1)

fault_counts = [0, 1, 5, 10, 25, 50, 100, 200, 500]
match_rates = []

for n_faults in fault_counts:
    matches = 0
    trials = 10  # Average over multiple fault injections
    for _ in range(trials):
        faulty = copy.deepcopy(model)
        if n_faults > 0:
            injector.inject_weight_faults(faulty, num_faults=n_faults)
        with torch.no_grad():
            preds = faulty(test_input).argmax(dim=1)
        matches += (preds == baseline_preds).float().mean().item()
    match_rates.append(matches / trials)
    print(f"Faults: {n_faults:>4d}  |  Prediction match rate: {match_rates[-1]:.2%}")

## Step 4: Visualize results

Plot the accuracy degradation curve.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fault_counts,
    y=[r * 100 for r in match_rates],
    mode='lines+markers',
    name='Prediction match rate',
    line=dict(color='#EF553B', width=2),
))
fig.update_layout(
    title='Accuracy Degradation Under Radiation Fault Injection',
    xaxis_title='Number of Bit-Flips Injected',
    yaxis_title='Prediction Match Rate (%)',
    yaxis_range=[0, 105],
    template='plotly_white',
)
fig.show()

## Step 5: Compare chip radiation hardness

Different chips have different SEU susceptibilities. Let's compare a commercial chip (Trillium TPU) vs a rad-hardened chip (BAE RAD5500).

In [ ]:
chips = {
    TRILLIUM_V6E.name: FaultInjector(rad_env=rad_env, chip_profile=TRILLIUM_V6E),
    RAD5500.name: FaultInjector(rad_env=rad_env, chip_profile=RAD5500),
}

for name, inj in chips.items():
    faults_per_orbit = inj.expected_faults_per_orbit(model, orbit)
    print(f"{name}: ~{faults_per_orbit:.1f} expected SEUs per orbit")

## Exercise

1. **Change the orbit altitude** to 1200 km (higher radiation). How does the fault rate change?
2. **Increase the model size** (add more layers/parameters). Does it become more or less resilient?
3. **Try INT8 quantization** using `space_ml_sim.compute.quantization.quantize_model()`. Are quantized models more resilient to bit-flips? Why?